# Divide and Conquer

In previous lectures, we saw two forms of recursion:

- **Linear recursion**: each call makes one recursive call (e.g. `find_max`, `reverse`, `binary_search`)
- **Nonlinear recursion**: each call makes two or more recursive calls (e.g. `count_nodes`, `height` on trees)

**Divide and conquer** is a general strategy that uses nonlinear recursion to solve problems on sequences (lists, arrays). It follows these steps:

1. **Simplest cases**: If the problem is small enough, solve it directly.
2. **Divide**: Split the problem into smaller subproblems (usually two halves).
3. **Conquer**: Solve each subproblem recursively using the same strategy.
4. **Combine**: Merge the solutions of the subproblems to form the solution to the original problem.

The key insight: by splitting the problem in half each time, we often achieve much better performance than processing elements one at a time.

---

## Binary Search as Divide and Conquer

We already saw binary search in Notebook 04. Let us now view it through the lens of divide and conquer.

**Problem**: Find whether `x` is in a sorted list `L`.

**Strategy**:
- **Simplest cases**: If the list is empty, return `False`.
- **Divide**: Compare `x` to the middle element. This divides the list into a left half and a right half.
- **Conquer**: Search in only one half (left if `x < L[mid]`, right if `x > L[mid]`).
- **Combine**: No combining needed — the answer comes directly from the subproblem.

Note: Binary search is a special case where we only recurse into **one** subproblem, not both.

In [ ]:
#
# Find x in sorted list using divide and conquer
#
def search(L, x, left, right):
    if left > right:
        return False
    mid = (left + right) // 2
    if x == L[mid]:
        return True
    elif x < L[mid]:
        return search(L, x, left, mid - 1)
    else:
        return search(L, x, mid + 1, right)

In [ ]:
A = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
print(search(A, 23, 0, len(A) - 1))
print(search(A, 50, 0, len(A) - 1))

---

## Fast Exponentiation

**Problem**: Compute $x^n$.

A naive approach multiplies $x$ by itself $n$ times, taking $n$ steps.

**Strategy using divide and conquer**:
- **Simplest cases**:
  + $x^0 = 1$
  + $x^1 = x$
- **Divide**: Split the exponent in half.
  + If $n$ is even: $x^n = x^{n/2} \times x^{n/2}$
  + If $n$ is odd: $x^n = x \times x^{n-1}$ (and $n-1$ is now even)


In [ ]:
#PID:24
#
# Your strategy in English: if b is 0 or 1, just return it.
# if n is even, use the same procedure to compute a = b^(n/2), and compute a*a
# if n is odd, use the same procedure to compute a = b^((n-1)/2), and compute b*a*a
#
def fast_exp(b, n):
    if n == 0:
        return 1
    if n == 1:
        return b
    a = fast_exp(b, n//2)
    if n%2==0:
        return a*a
    else:
        return b*a*a

fast_exp(3,3)

In [ ]:
#
# Compute x to the power n using divide and conquer
#
def fast_exp(x, n):
    if n == 0:
        return 1
    if n == 1:
        return x
    if n % 2 == 0:
        half = fast_exp(x, n // 2)
        return half * half
    else:
        half = fast_exp(x, n // 2)
        return x * half * half

In [ ]:
print(fast_exp(2, 10))    # 1024
print(fast_exp(3, 5))     # 243

**Why is this faster?**

To compute $x^{20}$:
- Naive: 20 multiplications
- Fast exponentiation: $x^{20} = (x^{10})^2$, and $x^{10} = (x^5)^2$, and $x^5 = x \times (x^2)^2$. Only about $\log_2 20 \approx 5$ multiplications.

**Important**: We compute `fast_exp(x, n // 2)` **once** and store it in `half`, rather than calling it twice. This is the difference between $\log n$ and $n$ steps.

Compare with this version that calls `fast_exp` twice:

In [ ]:
# This version is correct but inefficient!
def slow_exp(x, n):
    if n == 0:
        return 1
    if n == 1:
        return x
    if n % 2 == 0:
        return slow_exp(x, n // 2) * slow_exp(x, n // 2)
    else:
        return x * slow_exp(x, n // 2) * slow_exp(x, n // 2)

`slow_exp` computes the same subproblem **twice** at every step, which defeats the purpose of divide and conquer. We will revisit this issue in a later lecture on overlapping subproblems.

---


### Problem to solve: merge two sorted lists.

Input Examples: A = [1,10,20,25],  B = [5,7,21]

Output: [1,5,7,10,20,21,25]

Goal: come up with a strategy in English

Strategy: 
- Check if one or both is empty
  - If A is empty, output is B
  - If B is empty, output is A
- Else: both are not empty
  - Compare first elements of A and first element of B
  - (1) if A[0] > B[0], output is B[0] and the merge of the remaining lists.

A = [5, remaining of A]

B = [2, remaining of B]

if A[0] > B[0], output is "concatenation of sublist B[0], A[0], and result of the remaining A[1]..."



---

Things that must be done:
- Make the lists equal in size. [Not a good idea]
- Check if one or both is empty
  - If A is empty, output is B
  - If B is empty, output is A
- Compare first elements of A and B
- 


In [ ]:
#
# Output: a list (a merged/union of A and B, also sorted)
#
def merge(A, B):
    if A == []:
        return B
    if B == []:
        return A
    if A[0] > B[0]:
        return [B[0]] + merge(A, B[1:])
    else:
        return [A[0]] + merge(A[1:], B)

A = [1,10,20,25] 
B = [5,7,21]
merge(A,B)

#PID:25
---

```python
def merge(A, B):
    if A == []:
        return B
    if B == []:
        return A
    if A[0] > B[0]:
        return [B[0]] + merge(A, B[1:])
    else:
        return [A[0]] + merge(A[1:], B)
```

**Exercise: Describe this `merge` function in English (what it does, and at a high level (the core idea of) how it works)**

**Goal**: convey the strategy/idea of the algorithm **concisely** and **accurately**

**What it does**: combine two *sorted* lists into a sorted list.

**How it works**: concatenate the smaller of the two first items with the merge of the remaining items.


When we convey core ideas, sometimes we leave out operational details.



#PID:26
---

```python
def merge(A, B):
    if A == []:
        return B
    if B == []:
        return A
    if A[0] > B[0]:
        return [B[0]] + merge(A, B[1:])
    else:
        return [A[0]] + merge(A[1:], B)
```

**Exercise: Show at a high level how `merge([1,10,20,25], [5,7,21])` works.**

How it works: concatenate the smaller of the two first items with the merge of the remaining items.

1 smaller than 5: concatenate 1 and the merge of the remaining items.
- [1] + `merge([10,20,25], [5,7,21])`. And this is: [1] + [5,7,10,20,21,25]


Output is `[1,5,7,10,20,21,25]`, which is the expected correct output. So, this strategy/algorithm seems to work.


This is tracing the inputs.

```
merge([1,10,20,25], [5,7,21])
1 is smaller so we get [1] + merge([10,20,25], [5,7,21])
5 is smaller so we get [1,5] + merge([10,20,25], [7,21])
7 is smaller so we get [1,5,7] + merge([10,20,25], [21])
10 is smaller so we get [1,5,7,10] + merge([20,25], [21])
20 is smaller so we get [1,5,7,10,20] + merge([25], [21])
21 is smaller so we get [1,5,7,10,20,21] + merge([25], [])
B is empty so we just return A which gives us [1,5,7,10,20,21,25]



# take 1, merge ([10, 20, 25],[5,7,21])
# take 5, merge ([10, 20, 25],[7,21])
# take 7, merge ([10, 20, 25],[21])
# take 10, merge ([20, 25],[21])
# take 20, merge([25],[21])
# take 21, merge  ([25, []]
#appernd [25]
```

#PID:27

```python
L = [5,1,10,20,7,25,22,21]
```
Goal: sort L using `merge`. How?

If we (1) split L in two halves, (2) sort them, and (3) merge them, will this work?

Show how this can work with this example of L.

(1) Break L in two halves: L1 = [5,1,10,20], and L2 = [7,25,22,21]

(2) Sort them: L1 sorted = [1,5,10,20], L2 sorted = [7,21,22,25]




#PID:28


```python
L = [5,1,10,20,7,25,22,21]
```
Goal: sort L using `merge`. How?

If we (1) split L in two halves, (2) sort them, and (3) merge them, will this work?

Show how this can work with this example of L.

(1) Break L in two halves: L1 = [5,1,10,20], and L2 = [7,25,22,21]

(2) Sort them: L1 sorted = [1,5,10,20], L2 sorted = [7,21,22,25]

(3) Merge of [1,5,10,20] and [7,21,22,25] is [1,5,7,10,20,21,22,25]


Yes, it works for this example.


#PID:29
```python
L = [5,1,10,20,7,25,22,21]
```
Goal: sort L using `merge`.

Explain in one concise English sentence how to solve each of these 3 steps: (1) split L in two halves, (2) sort them, and (3) merge them.

(1) Break L in two halves: L1 = [5,1,10,20], and L2 = [7,25,22,21]

- How do we break L in two halves? Slice L from 0 to middle point (L1) and from middle point to the end (L2).

(2) Sort them: L1 sorted = [1,5,10,20], L2 sorted = [7,21,22,25]

- How do we sort L1 and L2? Call some sort function
   
(3) Merge of [1,5,10,20] and [7,21,22,25] is [1,5,7,10,20,21,22,25]

- How do we merge [1,5,10,20] and [7,21,22,25]? Call the merge function

  

My approach to problem solving (algorithms design, coding):
- first, we must have a solid strategy and plan
- think about it, be able to explain it in English
- be able to execute your idea/strategy in a programming language
  - make sure each step works

In [ ]:
#PID:30

#
# core idea: concatenate the smaller of the first elements of A and B, with the merge of the remaining items.
#
def merge(A, B):
    if A == []:
        return B
    if B == []:
        return A
    if A[0] > B[0]:
        return [B[0]] + merge(A, B[1:])
    else:
        return [A[0]] + merge(A[1:], B)

# core idea: (1) split L into 2 halves, (2) sort each half, (3) merge the sorted halves.

def my_sort(L):
    if len(L) <= 1:
        return L
    left, right = L[0: len(L)//2] , L[len(L)//2 : len(L)]
    sorted_left = my_sort(left)
    sorted_right = my_sort(right)
    return merge(sorted_left, sorted_right)
    

my_sort([10,5,20,30,15,7,1,12])

### How merge sort works step by step

Consider sorting `[38, 27, 43, 3, 9, 82, 10]`:

**Divide phase** (top-down):
```
[38, 27, 43, 3, 9, 82, 10]
       /                \
[38, 27, 43]        [3, 9, 82, 10]
   /     \            /         \
[38]  [27, 43]     [3, 9]    [82, 10]
       /   \        /  \       /   \
     [27] [43]    [3]  [9]  [82]  [10]
```

**Combine phase** (bottom-up, merging sorted sublists):
```
     [27] [43]    [3]  [9]  [82]  [10]
       \   /        \  /       \   /
[38]  [27, 43]     [3, 9]    [10, 82]
   \     /            \         /
[27, 38, 43]        [3, 9, 10, 82]
       \                /
[3, 9, 10, 27, 38, 43, 82]
```

If we don't trace recursive calls, here's what we conceptualize:
```
[38, 27, 43, 3, 9, 82, 10]
       /                \
[38, 27, 43]        [3, 9, 82, 10]
     | (magic)            | (magic)
[27, 38, 43]        [3, 9, 10, 82]
       \                /
[3, 9, 10, 27, 38, 43, 82]
```

### Where's the hard work in Merge Sort?

```python

def merge_sort(L):
    if len(L) <= 1:
        return L
    left, right = L[0: len(L)//2] , L[len(L)//2 : len(L)]
    sorted_left = merge_sort(left)
    sorted_right = merge_sort(right)
    return merge(sorted_left, sorted_right)    # <--- hard work
```

### A Different Way of Sorting

1. do the hard work to split the list L into two halves. (hard)
2. use the same sort procedure to sort each half. (easy)
3. then concatenate the sorted halves (easy)


```
L = [38, 27, 43, 3, 9, 82, 10]
```
Hint: (1) rearrange L into two lists with respect on an element, e.g. the first element (38).

We want to rearrange L in such a way that steps 2 and 3 will sort L.

How do we think we should rearrange L with respect to 38?

```
1. left, right = rearrange(L)
2. sorted_left = new_sort(left); 
   sorted_right = new_sort(right)
3. return sorted_left + sorted_right
```

In [ ]:
#PID:31
#
# Output: 2 lists A and B
# - A and B do not consist of the first element (38)
# - A = all elements less than first element (38)
# - B = all elements greater or equal to first element (38)
#
# how we do this:
# - go through each element of L starting from the second one
# -    if the element is smaller than the first, save it to A
# -    if the element is greater than or equal to the first, save it to B
# - return A and B
#
# we will translate this to Python at a high level
#  -- circle back to fill in details when needed
def rearrange(L):
    A, B = [], []
    for i in range(1, len(L)):
        if L[i] < L[0]:
            A.append(L[i])
        else:
            B.append(L[i])
    return A, B     

rearrange([38, 27, 43, 3, 9, 82, 10])

In [ ]:
#PID:32

# Complete the quick_sort function based on our strategy
#1. left, right = rearrange(L)
#2. sorted_left = new_sort(left); 
#   sorted_right = new_sort(right)
#3. return sorted_left + sorted_right

def rearrange(L):
    A, B = [], []
    for i in range(1, len(L)):
        if L[i] < L[0]:
            A.append(L[i])
        else:
            B.append(L[i])
    return A, B     


def quick_sort(L):
    if len(L) <= 1:
        return L
    left, right = rearrange(L)
    sorted_left = quick_sort(left)
    sorted_right = quick_sort(right)
    return sorted_left + [L[0]] + sorted_right
    

quick_sort([38, 27, 43, 3, 9, 82, 10])

### Merge sort vs Quick sort

| | Merge Sort | Quick Sort |
|:---|:---|:---|
| **Divide** | Simple: split in half | Hard: partition around pivot |
| **Combine** | Hard: merge two sorted lists | Simple: nothing to do |
| **Extra memory** | Yes (creates new lists) | No (sorts in place) |
| **Typical complexity** | $\Theta(n \log n)$ | $\Theta(n \log n)$ average |

---

## Summary: The Divide and Conquer Pattern

All divide-and-conquer algorithms follow the same template:

```python
def solve(problem):
    if problem is small enough:        # simplest case
        return direct_solution
    sub1, sub2 = divide(problem)       # divide
    sol1 = solve(sub1)                 # conquer
    sol2 = solve(sub2)                 # conquer
    return combine(sol1, sol2)         # combine
```

| Algorithm | Divide | Conquer | Combine |
|:---|:---|:---|:---|
| Binary search | Compare to middle | Search one half | None |
| Fast exponentiation | Split exponent in half | Compute $x^{n/2}$ | Square (and multiply by $x$) |
| Merge sort | Split list in half | Sort each half | Merge sorted halves |
| Quick sort | Partition around pivot | Sort each side | None |
| Tower of Hanoi | Separate top $n-1$ from bottom | Move $n-1$ disks | Move $n-1$ disks again |